# Tutorial 05: Wavelength Calibration

This tutorial turns the row index of an extracted spectrum into a wavelength,
and then measures how wide the spectrograph smears a line.

**The calibration is solved on a reference fiber, then propagated.** One fiber
gets a full search over every plausible assignment of arc lines to detected
peaks. Every other fiber starts from the solution of the fibre beside it, which
is far cheaper and keeps neighbouring solutions consistent.

**The synthetic arc has known wavelengths**, so the error of the recovered
solution can be measured instead of estimated.

**Before you start:** install the package and Jupyter in an environment that
has `numpy`, `astropy`, `matplotlib` and `scipy`; see `docs/CONTRIBUTING.md`.

## Imports

In [ ]:
# Uncomment for interactive pan and zoom (requires `pip install ipympl`).
# %matplotlib ipympl

import time
import warnings

import numpy as np
import matplotlib.pyplot as plt

from amasedrp.simulate import synthetic_fiber_flat, synthetic_arc
from amasedrp.reduction import identify_and_trace_fibers, extract_spectra
from amasedrp.calibration import (
    thar_lines,
    solve_wavelength_solution,
    apply_wavelength_solution,
    fit_line_spread_function,
)

## 1. The reference lines

A wavelength solution needs to know which arc lines to expect. The pipeline
carries a selection per channel, taken from the 2025-07 collimator-sweep
reductions. Three lists, used for three different jobs:

- `known_wls` — strong, well-measured lines. They **score** a candidate solution.
- `poss_wls` — the few strongest lines. They **enumerate** candidate assignments.
- `lsf_wls` — isolated lines, safe to fit a line profile to.

In [ ]:
lines = thar_lines("blue")

print(f"known lines      : {lines.known_wls.size}, "
      f"{lines.known_wls.min():.1f} - {lines.known_wls.max():.1f} A")
print(f"possible lines   : {lines.poss_wls.size}, "
      f"{lines.poss_wls.min():.1f} - {lines.poss_wls.max():.1f} A")
print(f"lines for LSF fit: {lines.lsf_wls.size}, {lines.lsf_wls}")
print()
print("The red channel carries its own list:")
red = thar_lines("red")
print(f"  red known lines: {red.known_wls.size}, "
      f"{red.known_wls.min():.1f} - {red.known_wls.max():.1f} A")

## 2. A synthetic arc with known wavelengths

The arc is built the way the instrument sees one: a lamp spectrum, multiplied
into the fiber flat, so the fibers carry the light and the gaps between blocks
stay dark.

The row-to-wavelength mapping is exactly linear here. A real spectrograph is
close to linear but not exactly, which is why the pipeline fits a polynomial of
degree 3 or more rather than a straight line.

In [ ]:
flat, true_centers = synthetic_fiber_flat(
    n_rows=9600, n_blocks=3, n_fibers_per_block=5,
)
arc, true_wavelength = synthetic_arc(
    flat,
    lines=lines.known_wls,
    wavelength_zero=4390.0,
    dispersion=0.08221,      # the dispersion measured on the instrument
    line_fwhm=0.49,          # the line width measured on the instrument
    noise_std=0.01,
    seed=11,
)

covered = (true_wavelength >= lines.known_wls.min()) & (
    true_wavelength <= lines.known_wls.max())
print(f"frame shape          : {arc.shape}")
print(f"row 0 wavelength     : {true_wavelength[0]:.2f} A")
print(f"row {arc.shape[0] - 1} wavelength   : {true_wavelength[-1]:.2f} A")
print(f"dispersion           : {true_wavelength[1] - true_wavelength[0]:.5f} A/row")
print(f"rows spanned by lines: {int(np.count_nonzero(covered))} of {arc.shape[0]}")

## 3. Identify, trace, extract

The arc is extracted exactly like any other frame. The trace model comes from
the flat, because the flat shows the fibers best.

In [ ]:
fibermap, tracemask = identify_and_trace_fibers(
    image=flat,
    n_blocks_expected=3,
    n_fibers_per_block_expected=5,
    poly_deg=3,
)
frame = extract_spectra(
    image=arc,
    tracemask=tracemask,
    fibermap=fibermap,
    method="boxcar",
    aperture_radius=3,
)

print(f"fibers : {frame.n_fibers}")
print(f"frame  : {frame.shape}")

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(frame.wave, frame.flux[7], lw=0.7)
ax.set_xlabel("row (pixel)")
ax.set_ylabel("flux [ADU]")
ax.set_title("Extracted arc spectrum, reference fiber")
plt.show()

## 4. Solve the wavelength solution

`solve_wavelength_solution` picks the fiber with the longest trace and scores
every plausible assignment of the strongest possible lines to the strongest
detected peaks. That search is the expensive part of the whole calibration, and
it is worth knowing its size: with 20 candidate peaks, 7 possible wavelengths,
and a polynomial degree free to run from 4 to 7, this frame enumerates
**843 999 candidate assignments**.

Everything after that is cheap. The remaining fibers start from the solution of
the already-solved neighbour and refine it by least squares, which is a few tens
of milliseconds each.

In [ ]:
started = time.perf_counter()
solution = solve_wavelength_solution(
    frame,
    tracemask,
    known_wls=lines.known_wls,
    poss_wls=lines.poss_wls,
    min_deg=3,
    poly_kind="legendre",
    full_search=True,
)
elapsed = time.perf_counter() - started

print(f"elapsed                : {elapsed:.1f} s")
print(f"fibers with a solution : {solution.calibrated_fraction:.1%}")
print(f"scores                 : median {np.nanmedian(solution.scores):.5f}, "
      f"max {np.nanmax(solution.scores):.5f}")
print(f"fiber 7 polynomial     : degree {solution.coeffs.shape[1] - 1}")

A score of `-1` marks a fiber that could not be solved and inherited its
neighbour's solution instead. On this frame no fiber fell back, so every score
is a real fit. The score itself is a residual: a smaller number means the
solution reproduces the known lines better.

For comparison, the same search on the real blue arc frames of the 2025-07
sweep returns a score of 0.019. The synthetic frame has a median score of
0.0166, and 0.0146 on the reference fiber, which is the same order, so this
frame exercises the solver the way the real data does.

## 5. Apply the solution

`apply_wavelength_solution` replaces the frame's `wave` array with a
per-fiber wavelength array, and records the score of every fiber in the
`FiberMap` as `WAVCAL_SCORE`.

In [ ]:
calibrated = apply_wavelength_solution(frame, solution)

print(f"wave shape   : {calibrated.wave.shape}   (n_fibers, n_wave)")
print(f"frame shape  : {calibrated.flux.shape}")
print(f"row 0        : {calibrated.wave[0, 0]:.2f} A on fiber 0, "
      f"{calibrated.wave[7, 0]:.2f} A on fiber 7")
print(f"new column   : {calibrated.fibermap.colnames[-1]}")
print(f"score spread : {np.nanmin(calibrated.fibermap['WAVCAL_SCORE']):.5f} - "
      f"{np.nanmax(calibrated.fibermap['WAVCAL_SCORE']):.5f}")

Fibers that are solved independently must agree where they overlap: the lamp is
the same for all of them. The spread across fibers is a check that the
propagation worked.

In [ ]:
row = 4000
across_fibers = calibrated.wave[:, row]
print(f"at row {row}: wavelength across 15 fibers")
print(f"  spread : {np.max(across_fibers) - np.min(across_fibers):.4f} A")
print(f"  mean   : {np.mean(across_fibers):.3f} A   (true {true_wavelength[row]:.3f} A)")

## 6. Check the solution against the truth

This is the check only a synthetic frame can give. The recovered wavelength is
compared against the wavelength that was injected.

In [ ]:
rows = np.arange(frame.n_wave)
recovered = solution.eval(rows)[0]
error = recovered - true_wavelength

line_rows = np.round(
    (lines.known_wls - true_wavelength[0]) / 0.08221).astype(int)
low, high = int(line_rows.min()), int(line_rows.max())
inside = (rows >= low) & (rows <= high)

print(f"known lines cover rows {low} - {high} "
      f"({true_wavelength[low]:.1f} - {true_wavelength[high]:.1f} A)")
print()
print(f"  inside the covered range : rms {np.sqrt(np.mean(error[inside] ** 2)):.4f} A, "
      f"max {np.max(np.abs(error[inside])):.4f} A")
print(f"  outside, extrapolated    : rms {np.sqrt(np.mean(error[~inside] ** 2)):.4f} A, "
      f"max {np.max(np.abs(error[~inside])):.4f} A")
print()
worst = np.argsort(np.abs(error[line_rows]))[::-1][:5]
print("worst 5 known lines:")
for position in np.sort(worst):
    row = line_rows[position]
    print(f"  {lines.known_wls[position]:9.3f} A (row {row:5d}): "
          f"recovered {recovered[row]:9.3f}, error {error[row]:+8.4f} A")

Read the two numbers together, because the gap between them is the point.

Inside the range the arc lines cover, the solution reproduces the truth to an
rms of 0.08 A. That is a sixth of a resolution element, so the calibration is
not the limiting factor there.

Outside that range the polynomial is extrapolating. It still returns a number,
and the number is still smooth, but the error grows to 0.7 A rms and 1.2 A at
the worst row. Nothing in the pipeline flags this: the rows are labelled with a
wavelength and look perfectly ordinary. **A wavelength array is only as good as
the arc lines that constrain it**, and the coverage has to be checked
separately.

## 7. The acceptance fraction

A focus sweep and a real observation must be treated differently, and the
pipeline makes that explicit.

At strong defocus the arc lines broaden and blend, and most fibers legitimately
fail to solve. That is expected during a sweep. During a real observation it is
not: the lamp does not care about the telescope pointing, so a shortfall points
at a software or hardware fault and the observer has to be told.

`solve_wavelength_solution(min_calibrated_fraction=...)` turns that into a hard
failure. The threshold is validated before any work is done, so a wrong value
costs nothing.

In [ ]:
for bad in (1.5, -0.1):
    try:
        solve_wavelength_solution(
            frame, tracemask,
            known_wls=lines.known_wls, poss_wls=lines.poss_wls,
            min_calibrated_fraction=bad,
        )
    except ValueError as error:
        print(f"min_calibrated_fraction={bad}: ValueError: {error}")

print()
print(f"this frame solved {solution.calibrated_fraction:.1%} of its fibers, "
      f"so a threshold of 0.95 would pass and a frame at 60% would raise "
      f"RuntimeError instead.")

## 8. The line-spread function

The line-spread function says how much the spectrograph widens a line. It is
measured from the arc itself: for every fiber and every isolated line, the
pipeline cuts a window around the line, centres it on the nearest detected
peak, and fits a Gaussian.

The injected lines have a 0.49 A FWHM, which is the width measured on the real
frames. The recovered width is compared against that.

In [ ]:
lsf = fit_line_spread_function(calibrated, lines.lsf_wls)

print(f"target wavelengths : {lsf.target_wls}")
print(f"fwhm shape         : {lsf.fwhm.shape}   (n_wls, n_fibers)")
print(f"valid fibers       : {int(lsf.valid.sum())} of {lsf.fwhm.size}")
print()
for index, target in enumerate(lsf.target_wls):
    width = np.nanmedian(lsf.fwhm[index])
    print(f"  {target:8.3f} A : fwhm {width:.4f} A "
          f"({width / 0.49:.3f} x injected), R = {target / width:.0f}")

print()
print(f"dispersion is 0.08221 A/row, so {np.nanmedian(lsf.fwhm) / 0.08221:.2f} rows per FWHM")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.4))

for index in range(lsf.fwhm.shape[0]):
    axes[0].plot(np.arange(lsf.fwhm.shape[1]), lsf.fwhm[index],
                 "o", ms=4, label=f"{lsf.target_wls[index]:.1f} A")
axes[0].axhline(0.49, color="k", ls="--", lw=0.8, label="injected 0.49 A")
axes[0].set_xlabel("fiber index")
axes[0].set_ylabel("fwhm [A]")
axes[0].set_title("Line-spread function per fiber")
axes[0].legend(fontsize=8)

window = 8
for target in lines.lsf_wls:
    column = int(np.argmin(np.abs(calibrated.wave[7] - target)))
    axis = calibrated.wave[7, column - window:column + window + 1]
    flux = calibrated.flux[7, column - window:column + window + 1]
    axes[1].plot(axis, flux / np.nanmax(flux), lw=1.0, label=f"{target:.1f} A")
axes[1].set_xlabel("wavelength [A]")
axes[1].set_ylabel("normalised flux")
axes[1].set_title("Arc lines, fiber 7")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

The width comes back at 1.00 times what was injected, and the resolution is
9469 to 10139 across the three lines. This matches the instrument: about six
rows per FWHM at H-alpha, which is the number recorded in the design notes.

Two things to watch when you use this on real data:

- **Isolated lines only.** A pair of strong lines inside the fitting window is
  rejected and returns `nan`, rather than fitting a blend and reporting a width
  for it. `nan` is a result, not a failure to hide.
- **The window is narrow.** `lsf_fitting` centres the cut-out on the nearest
  detected peak, so it measures the line the peak belongs to. If two candidates
  are close, check the returned width against the neighbouring fibers.

## Summary

| Step | Call | Result |
| --- | --- | --- |
| Reference lines | `thar_lines("blue")` | `known_wls`, `poss_wls`, `lsf_wls` |
| Solve | `solve_wavelength_solution(frame, tracemask, ...)` | `WavelengthSolution` |
| Apply | `apply_wavelength_solution(frame, solution)` | `FiberFrame` with per-fiber `wave` |
| Line width | `fit_line_spread_function(frame, lsf_wls)` | `LineSpreadFunction` |

Measured on this synthetic frame:

- 843 999 candidate assignments scored for the reference fiber, about 18 s; the
  other 14 fibers are three orders of magnitude cheaper each.
- Every fiber solved, median score 0.0166, against 0.019 on the real blue
  frames.
- Recovered wavelength error: 0.08 A rms inside the arc-line range, 0.72 A rms
  outside it, where the polynomial extrapolates.
- Line-spread function: 0.49 A injected, 0.49 A recovered, resolving power
  9469 to 10139.

**Where to go next:** tutorial 07 runs preprocessing, extraction and
calibration as one pipeline.